# 02. Practice: KV 캐시와 통합 출력 포맷 실험

## 목표

이 노트북은 두 가지 실무 감각을 기릅니다.

1. 장문 OCR에서 디코더 KV 캐시가 왜 병목이 되는지 계산합니다.
2. 서로 다른 비전 태스크를 텍스트/이미지 생성 포맷으로 바꾸는 연습을 합니다.

## 실행 방법

Python 표준 라이브러리만 사용합니다. 숫자는 실제 모델의 정확한 메모리 사용량이 아니라 구조를 이해하기 위한 근사치입니다.


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

@dataclass
class Paper:
    """논문 메타데이터를 담는 작은 자료형입니다.
    
    dataclass를 쓰면 dict보다 필드가 명확해집니다.
    학습 자료에서는 구조를 드러내는 것이 코드 길이보다 중요할 때가 많습니다.
    """
    title: str
    category: str
    main_bottleneck: str
    practical_question: str

papers = [
    Paper("Scalable Visual Pretraining", "visual_pretraining", "OCR 전처리 손실", "원본 페이지 이미지를 언제 보존해야 할까?"),
    Paper("Unlimited OCR", "long_ocr", "출력 KV 캐시 증가", "긴 문서를 몇 페이지까지 한 번에 처리할 수 있을까?"),
    Paper("SenseNova-Vision", "unified_generation", "태스크별 헤드 파편화", "출력 포맷을 하나의 API로 통일할 수 있을까?"),
    Paper("LingBot-Vision", "spatial_perception", "경계/깊이 정보 부족", "공간 지각 성능을 어떻게 따로 평가할까?"),
    Paper("LingBot-World 2.0", "world_model", "긴 상호작용과 실시간성", "행동 루프에서 지연을 어떻게 줄일까?"),
]

for paper in papers:
    print(f"{paper.title:<28} | {paper.main_bottleneck} | {paper.practical_question}")


## 1. KV 캐시 메모리 근사

일반적인 decoder-only transformer에서 KV 캐시는 대략 다음에 비례합니다.

`2 * layers * sequence_length * hidden_size * bytes_per_value`

`2`는 key와 value 두 텐서를 뜻합니다. 실제 구현에서는 batch, tensor parallel, head dimension, padding, allocator overhead가 더해집니다.


In [ ]:
def kv_cache_gib(layers: int, hidden_size: int, seq_len: int, bytes_per_value: int = 2) -> float:
    """KV 캐시 크기를 GiB 단위로 근사합니다.
    
    bytes_per_value=2는 fp16/bf16 저장을 가정합니다.
    이 함수는 정확한 벤치마크가 아니라 길이에 따른 증가율을 보기 위한 계산입니다.
    """
    total_bytes = 2 * layers * hidden_size * seq_len * bytes_per_value
    return total_bytes / (1024 ** 3)

model_config = {
    "layers": 32,
    "hidden_size": 4096,
    "bytes_per_value": 2,
}

for seq_len in [1024, 4096, 8192, 16384, 32768]:
    memory = kv_cache_gib(seq_len=seq_len, **model_config)
    print(f"seq_len={seq_len:>5} -> KV cache ~= {memory:5.2f} GiB")


## 2. Full attention과 R-SWA 스타일 창 비교

R-SWA의 핵심 직관은 모든 과거 출력 토큰을 저장하지 않고, 참조 토큰과 최근 출력 창을 유지하는 것입니다. 아래 코드는 실제 R-SWA 구현이 아니라 메모리 증가 모양을 비교하는 toy model입니다.


In [ ]:
def full_cache_tokens(reference_tokens: int, generated_tokens: int) -> int:
    # 일반적인 방식에서는 참조 입력과 지금까지 만든 출력이 모두 attention 범위에 남습니다.
    return reference_tokens + generated_tokens

def sliding_cache_tokens(reference_tokens: int, generated_tokens: int, window: int) -> int:
    # 슬라이딩 방식에서는 참조 입력은 유지하고, 출력은 최근 window개만 유지한다고 가정합니다.
    return reference_tokens + min(generated_tokens, window)

reference_tokens = 512
window = 256
print("generated | full tokens | sliding tokens")
print("----------|-------------|---------------")
for generated in [0, 256, 1024, 4096, 16384, 32768]:
    print(f"{generated:>9} | {full_cache_tokens(reference_tokens, generated):>11} | {sliding_cache_tokens(reference_tokens, generated, window):>13}")


In [ ]:
def attention_view(step: int, reference_count: int = 4, window: int = 3) -> Dict[str, List[str]]:
    """특정 생성 step에서 어떤 토큰을 본다고 가정하는지 보여줍니다.
    
    reference token은 이미지 패치나 프롬프트처럼 계속 참조해야 하는 입력입니다.
    output token은 이미 생성한 문서 전사 결과입니다.
    """
    refs = [f"ref{i}" for i in range(reference_count)]
    outputs = [f"out{i}" for i in range(step)]
    visible_outputs = outputs[-window:]
    return {"reference": refs, "recent_outputs": visible_outputs}

for step in range(1, 8):
    view = attention_view(step)
    print(f"step={step}: refs={view['reference']} recent={view['recent_outputs']}")


## 3. 통합 멀티모달 생성 출력 만들기

SenseNova-Vision 계열의 핵심은 서로 다른 비전 태스크를 한 모델의 출력 규약으로 바꾸는 것입니다. 아래 예시는 detection, segmentation, depth를 아주 작은 포맷으로 시리얼라이즈합니다.


In [ ]:
import json

def serialize_detection(label: str, bbox: Tuple[int, int, int, int]) -> str:
    # 좌표는 텍스트로 생성할 수 있지만, 후처리기가 읽을 수 있게 JSON 규약을 정합니다.
    return json.dumps({"task": "detect", "objects": [{"label": label, "bbox_xyxy": bbox}]}, ensure_ascii=False)

def serialize_segmentation(mask: List[List[int]]) -> str:
    # 실제 모델은 이미지를 출력할 수 있지만, toy 예제에서는 0/1 마스크를 문자열로 둡니다.
    rows = ["".join("#" if value else "." for value in row) for row in mask]
    return "\n".join(rows)

def serialize_depth(depth: List[List[float]]) -> str:
    # 깊이 맵도 dense output입니다. 여기서는 소수점 한 자리로 압축해 봅니다.
    rows = [" ".join(f"{value:.1f}" for value in row) for row in depth]
    return "\n".join(rows)

print("[detection]")
print(serialize_detection("table", (10, 20, 180, 90)))

print("\n[segmentation mask]")
print(serialize_segmentation([
    [0, 0, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 0, 0],
]))

print("\n[depth map]")
print(serialize_depth([
    [2.0, 2.1, 2.2],
    [1.4, 1.5, 1.7],
    [0.9, 1.0, 1.2],
]))


## 4. 생성 결과 검증기

통합 생성 모델은 출력 형식이 깨질 수 있습니다. 그래서 실무에서는 모델만큼 검증기가 중요합니다.


In [ ]:
def validate_detection_json(text: str) -> bool:
    """detection 출력이 최소 규약을 만족하는지 검사합니다."""
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        return False
    if payload.get("task") != "detect":
        return False
    objects = payload.get("objects")
    if not isinstance(objects, list) or not objects:
        return False
    for obj in objects:
        bbox = obj.get("bbox_xyxy")
        if not isinstance(bbox, list) or len(bbox) != 4:
            return False
        if not all(isinstance(x, int) for x in bbox):
            return False
    return True

valid = serialize_detection("table", (10, 20, 180, 90))
invalid = "detect table at 10 20 180 90"

print("valid output:", validate_detection_json(valid))
print("invalid output:", validate_detection_json(invalid))


## 정리

- 장문 OCR은 입력 길이보다 출력 길이 때문에 메모리 문제가 커질 수 있습니다.
- 슬라이딩 창 방식은 메모리 증가를 막지만, 멀리 떨어진 의존성을 어떻게 다룰지가 핵심 평가 포인트입니다.
- 통합 생성형 비전 모델은 출력 포맷을 통일할수록 편해지지만, 검증기와 후처리기가 반드시 필요합니다.
